# INTEGRATES Building-Energy Calibration Challenge

## 00 - Challenge contract

Your task is to select a subset of the twelve available calibration parameters and replace the search strategy in the repository-root `calibration.py`. The search space contains uncertain operating schedules, controls, and tightly bounded heat-loss multipliers; audited values remain the fixed defaults. A two-regime model represents normally heated space and wings/floors operated at lower temperatures. Bolzano's legal heating season and dated exceptions are fixed challenge inputs, not calibration variables. Do not change the challenge-owned simulation, inputs, measurements, metrics, or export code.

Your file must define:

- `TEAM_NAME: str`
- `ALGORITHM_NAME: str`
- `calibrate(objective, parameter_space, budget, seed) -> dict[str, float]`

The returned dictionary must contain every supplied parameter name exactly once, with finite values inside the stated bounds. Call `objective(vector)` for a sequential evaluation. Methods that naturally propose a population may call `objective.evaluate_many(vectors)`; both interfaces use the same challenge-controlled local process pool, cache, validation, and evaluation budget. The vector order is the order of `parameter_space`. Every process runs on the laptop executing this notebook—there is no central simulation server or cross-laptop process pool. The official scored run uses buildings 01–03, years 2018–2019, January warm-up, and February/March scoring.


## 01 - Imports & Initialization

In [ ]:
import integrates_hackathon.notebook_workflow as workflow

# Official scoring configuration. Teams may temporarily shorten this list
# while developing, but final verification restores both official years.
CALIBRATION_YEARS = [2018, 2019] # The challenge will be judged on February and March of 2018 and 2019; during development you could reduce to just one year for faster iterations.
EVALUATION_BUDGET = 48  # The contest will be scored with budget of 48, but during development you could reduce this for faster iterations.
PARALLEL_WORKERS = 12  # Reduce this if the laptop has fewer available cores.

context = workflow.initialize_challenge(
    backend="pbe",
    building_ids=("building_01", "building_02", "building_03"),
    # building_ids=("building_02",), ## During development, you may wish to calibrate only one building for faster iterations --> recommend building_02
    calibration_years=tuple(CALIBRATION_YEARS),
    score_months=(2, 3),
    warmup_month=1,
    evaluation_budget=EVALUATION_BUDGET,
    parallel_workers=PARALLEL_WORKERS,
    seed=2026,
)
workflow.show_challenge_contract(context)


## 02 - Visualize Input Data

This section parses inputs for `building_01`, `building_02`, and `building_03` and displays the February/March bills and 2018–2019 weather. Fixed audited properties and the calibration bounds remain visible. Bolzano's legal heating season and dated exceptions are fixed inputs, not variables teams may change.


In [ ]:
input_views = workflow.show_input_data(context)

## 03 - Load Baseline Energy Simulation

The notebook loads organizer-precomputed exact-window baseline results. Each selected year simulates every hour of January, February, and March; January is thermal warm-up and is discarded. Only February and March are compared with bills, with no annualization. Challengers do not rerun the baseline.

In [ ]:
baseline = workflow.load_baseline(context)
baseline_results = workflow.show_baseline_results(context, baseline)

## 04 - Run Calibration

This section imports and runs the repository-root `calibration.py`. The shipped starter evaluates the default vector once and returns it unchanged, so its calibrated score exactly matches the baseline until participants replace the passthrough with a search strategy. Each laptop creates its own local worker pool; no simulations or data leave that computer. Independent weather years run in separate processes. Participant methods may use the optional `objective.evaluate_many(vectors)` interface when they can propose candidates together. Before each exact-window task, the parent process prints the building, scored month/year, and calibration-set counter.

The workflow passes `parameter_space`, `budget`, and `seed` into `calibration.py`. Each item in `parameter_space` provides `.name`, `.lower`, `.upper`, and `.default`; participant code should read these values rather than duplicate bounds or assume a particular budget. Teams may use larger local budgets while developing, but final verification runs their unchanged `calibration.py` in a clean challenge copy with the official budget, buildings, years, months, seed, and parameter contract. The objective stops with an error if code requests more than the permitted number of new candidates.

A submission returns one scalar value per active parameter and building. That same value is applied to every scored month and year. Keys containing special months/years, nested schedules, and extra return keys are rejected. Final verification transfers `calibration.py` plus the active-parameter selection recorded in the standardized JSON; edits to simulation, inputs, scoring, or notebook files do not accompany a submission.


In [ ]:
### Calibratable parameters ###
# Replace None with an in-range numeric starting value to activate a parameter.
# None holds that parameter at the organizer default shown in its comment.
calibratable_params_init = {
    "weekday_heating_hours": None, # active occupied-setpoint hours per weekday; default 8; simulator rounds to a whole hour ## allowable range [4,12]
    "non_workday_heating_hours": None, # active hours on weekends/public holidays; default 2; simulator rounds to a whole hour ## allowable range [0,4]
    "weekday_start_hour": None, # start of active weekday heating; default 7; simulator rounds to a whole hour ## allowable range [5,9]
    "occupied_heating_setpoint_c": None, # normal-regime setpoint during active hours; default 20 ## allowable range [18.5,21.5]
    "night_setback_temperature_c": None, # maintained setpoint outside active hours, with energy counted; default 15 ## allowable range [12,18]
    "outdoor_shutdown_temperature_c": None, # 24-hour mean outdoor threshold switching active heating to setback; default 15 ## allowable range [12,18]
    "normally_heated_area_fraction": None, # conditioned area following normal rather than reduced regime; default 0.8 ## allowable range [0.6,1.0]
    "reduced_area_setpoint_c": None, # active setpoint for reduced-regime areas; default 15 ## allowable range [12,17]
    "reduced_area_schedule_fraction": None, # reduced active hours divided by normal active hours; default 0.5 ## allowable range [0.25,0.75]
    "air_exchange_multiplier": None, # multiplier on audited/default infiltration and ventilation; default 1.0 ## allowable range [0.7,1.3]
    "opaque_envelope_u_value_multiplier": None, # common audited wall/roof/floor U-value multiplier; default 1.0 ## allowable range [0.9,1.1]
    "window_u_value_multiplier": None, # audited window U-value multiplier; default 1.0 ## allowable range [0.9,1.1]
}

# Organizer-owned assertions stop the run if keys change or if any activated
# starting value is non-numeric or outside its organizer-owned range.
context = workflow.configure_parameter_contract(
    context, calibratable_params_init
)

calibration_run = workflow.run_calibration(context)
calibrated_parameters = workflow.show_calibration_results(context, calibration_run)

## 05 - Validate & Compare Results

The organizer-owned workflow validates every returned value and compares the independently rerun calibrated simulation with the baseline and measured monthly consumption.

In [ ]:
comparison_results = workflow.show_calibrated_comparison(
    context, baseline, calibration_run
)

## 06 - Scoring

WAPE is calculated only from the February/March bills in `CALIBRATION_YEARS`; nothing is annualized. The example positive score is `max(0, 100 - WAPE%)`. Each building is weighted equally so a large building cannot dominate. Baseline score, improvement, CV(RMSE), NMBE, and evaluation count remain visible for interpretation and possible tie-breaking.

In [ ]:
score_report = workflow.score_and_visualize(context, baseline, calibration_run)

## 07 - Export

Every team produces the same four artifacts: a one-row-per-building summary, a one-row-per-building-month comparison, an organizer-recorded trace of every unique budget-charged evaluation, and a machine-readable JSON result containing metadata, scores, active parameter selection, the calibration.py SHA-256 hash, and calibrated parameters.

In [ ]:
exported_paths = workflow.export_standardized_results(context, calibration_run, score_report)
workflow.show_exported_results(exported_paths)